In [ ]:
import os
import pandas as pd
import cv2
from glasses_detector import GlassesClassifier

clf = GlassesClassifier()

root_path = "kagglehub/datasets/jeffheaton/glasses-or-no-glasses/versions/2/"
df = pd.read_csv(root_path + "train.csv")

corrected_images = {
    "image-no": [],
    "final-label": [],
}

prefix = "face-"
labels = ["no-glasses", "glasses"]
images = root_path + "faces-spring-2020/faces-spring-2020"

for idx, row in df.iterrows():
    label = int(row["glasses"])
    img_name = prefix + f"{idx + 1}.png"
    img_path = os.path.join(images, img_name)

    img = cv2.imread(img_path)
    

    predicted = clf.predict(image=img_path, format="int")

    if label != predicted:
        print(f"\nMismatch at {idx+1}")
        print(f"Predicted: {predicted}, Actual: {label}")

        display = img.copy()
        cv2.putText(display, f"Label: {labels[label]}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        # cv2.putText(display, f"Pred: {labels[predicted]}", (20, 80), # prediction by the classifier
        #             cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.imshow("Mismatch", display)

        key = cv2.waitKey(0)

        if key == ord('q'):
            break

        elif key == ord('m'):  # flip label
            new_label = 1 - label
            df.at[idx, "glasses"] = new_label

            corrected_images["image-no"].append(idx + 1)
            corrected_images["final-label"].append(new_label)


          

        cv2.destroyAllWindows()

    if idx % 100 == 0:
        df.to_csv("train-corrected.csv", index=False)
       
# Final save
df.to_csv("train-corrected.csv", index=False)


cdf = pd.DataFrame(corrected_images)
cdf.to_csv("correction-log.csv", index=False)





ImportError: cannot import name 'GlassesClassifier' from 'glasses_detector' (c:\Users\arushaarya\miniconda3\envs\py310\lib\site-packages\glasses_detector\__init__.py)

In [ ]:
#dataclass for dependency injection
from torch.utils.data import Dataset
import pandas as pd
import os
import cv2
import torch


class TrainDataset(Dataset):
    def __init__(self,dir_path,csv_path,transformation=None):
        self.prefix = "face-"
        self.dir = dir_path
        self.df = pd.read_csv(csv_path)
        self.transform = transformation
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        label = int(self.df.iloc[index]["glasses"])
        img_name = self.prefix + f"{index + 1}.png"
        img_path = os.path.join(self.dir, img_name)

        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # normalize to [0,1]
        img = img / 255.0
       
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        img = self.transform(img) if self.transform else img
        return img,label
    

#dataloader
from torch.utils.data import DataLoader

dataset = TrainDataset(
    dir_path="resized_dataset",
    csv_path="train-corrected.csv"
)

loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [ ]:
torch.set_float32_matmul_precision('high')

In [7]:
#VAE class

import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),   # 64 → 32
            nn.LeakyReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),  # 32 → 16
            nn.LeakyReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), # 16 → 8
            nn.LeakyReLU(),
        )

        self.fc_mu = nn.Linear(128 * 8 * 8, z_dim)
        self.fc_logvar = nn.Linear(128 * 8 * 8, z_dim)

        # Decoder
        self.fc_dec = nn.Linear(z_dim, 128 * 8 * 8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 8 → 16
            nn.LeakyReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 16 → 32
            nn.LeakyReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),    # 32 → 64
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.fc_dec(z)
        x = x.view(-1, 128, 8, 8)
        return self.decoder(x)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar



def vae_loss(recon_x, x, mu, logvar, beta=0.05):
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='mean')

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + beta * kl





In [ ]:
#train VAE
from torchvision.utils import save_image
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
lr = 1e-3
loss_type = "BCE"
activation="leakyRELU"
z_dim=64
beta=0.05
model = VAE(z_dim=z_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
os.makedirs(f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}",exist_ok=True)
for epoch in range(80):
    model.train()
    total_loss = 0
    for imgs, _ in loader:
        imgs = imgs.to(device)

        recon, mu, logvar = model(imgs)
        loss = vae_loss(recon, imgs, mu, logvar,beta)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
      # 16 images, z_dim=64
    with torch.no_grad():
        z = torch.randn(1, z_dim).to(device)*1.2 
        generated = model.decode(z)[0].cpu().permute(1,2,0)
    
    generated = generated.cpu().numpy()*255
    
    generated = generated.astype("uint8")
    img = cv2.cvtColor(generated,cv2.COLOR_RGB2BGR)
    # save = cv2.resize(img,(256,256),interpolation=cv2.INTER_NEAREST)
    # save_image(generated,f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png")
    cv2.imwrite(f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png",img)
    
    
    # save_image(save,f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png")


    print(f"Epoch {epoch}, Loss: {total_loss}")

cuda
Epoch 0, Loss: 47.00581955909729
Epoch 1, Loss: 44.89797621965408
Epoch 2, Loss: 44.601789593696594
Epoch 3, Loss: 44.37574976682663
Epoch 4, Loss: 44.188894510269165
Epoch 5, Loss: 44.05966794490814
Epoch 6, Loss: 43.960757195949554
Epoch 7, Loss: 43.89940309524536


KeyboardInterrupt: 

In [9]:
#GAN

import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()

        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0),  # 1 → 4
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),   # 4 → 8
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),   # 8 → 16
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),    # 16 → 32
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),      # 32 → 64
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)
    




class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),   # 64 → 32
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1), # 32 → 16
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1),# 16 → 8
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 2, 1),# 8 → 4
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Conv2d(512, 1, 4, 1, 0),  # → 1
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)
    




    

In [10]:
GAN_dataset = TrainDataset("resized_dataset","train-corrected.csv",transformation=lambda x: (x-0.5)/0.5)
GAN_Loader = DataLoader(GAN_dataset,batch_size=64,shuffle=1)


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

z_dim = 100

G = Generator(z_dim).to(device)
D = Discriminator().to(device)

optimizer_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.999))

criterion = nn.BCELoss()

True


In [38]:

epochs = 50
import torchvision.utils as vutils
os.makedirs("generated/GAN",exist_ok=True)
for epoch in range(epochs):
    for imgs, _ in GAN_Loader:
        imgs = imgs.to(device)
        batch_size = imgs.size(0)

        # labels
        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        # =========================
        # Train Discriminator
        # =========================
        optimizer_D.zero_grad()

        # real images
        real_output = D(imgs)
        loss_real = criterion(real_output, real_labels)

        # fake images
        z = torch.randn(batch_size, z_dim, 1, 1, device=device)
        fake_imgs = G(z)

        fake_output = D(fake_imgs.detach())
        loss_fake = criterion(fake_output, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # =========================
        # Train Generator
        # =========================
        optimizer_G.zero_grad()

        output = D(fake_imgs)
        loss_G = criterion(output, real_labels)

        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")
    with torch.no_grad():
        z = torch.randn(1, z_dim, 1, 1, device=device)
        samples = G(z).cpu().numpy()

    # convert from [-1,1] → [0,1]
        samples = (samples * 0.5) + 0.5
        samples = samples*255
        img = cv2.cvtColor(samples,cv2.COLOR_RGB2BGR)
        cv2.imwrite(f"generated/GAN/gan_epoch_{epoch+1}.png",samples)
        # vutils.save_image(samples, f"generated/GAN/gan_epoch_{epoch+1}.png", nrow=4)



RuntimeError: Given transposed=1, weight of size [100, 512, 4, 4], expected input[64, 64, 1, 1] to have 100 channels, but got 64 channels instead